# 🎬 Whose Dime — assembly

**One video, from its Drive folder to `render/video.mp4`.** The notebook reads `03_scenes.md`, the images, the Flow clips and the voice (`voice/voice.wav`, `voice/words.json`, made by the voice notebook), draws the charts from `02b_factcheck.md`, and writes in `render/`:

| File | What it holds |
|---|---|
| `video.mp4` | the whole video, 1920x1080, 30 fps, sound at -14 LUFS |
| `chapters.txt` | the chapter times for the YouTube description, `0:00` first |
| `verdict_video.json` | the short verdict: `ready`, the technical check, the warnings, the code (`main@<commit>`) |
| `report_video.json`, `preview_video/` | every scene's start and length; six frames to look at |
| `extract_001-010.mp4` | with `MODE = "extract"`: its own `verdict_extract_001-010.json` and `report_`, `preview_`; **the whole video's verdict is never touched** |

All the numbers (card length, motions, loudness, the check's thresholds) are in `whosedime/whosedime-video/settings.json` in the repo; the colours in `whosedime/common/palette.json`.

## How to use

1. Fill in the form, then **Runtime → Run all**, and allow Google Drive. No GPU needed.
2. **First, an extract**: `MODE = "extract"`, `SCENES = "1-10"` (about 5 min). Look at it, and at `render/preview_extract_001-010/`.
3. **Then the whole video**: `MODE = "full"` (about 30-45 min). Then ask Claude to read `render/verdict_video.json`.
4. `MODE = "plan"` checks everything (gate, charts, timing, chapters) without rendering: a minute.

**The gate (decision D4)**: the assembly refuses to start if the checker blocks the scenes or the voice version (a [TO VERIFY] fact used, a registry changed since Franco signed it, scenes that are no longer the script), or if the voice is not `ready`. A chart citing a fact that is not verified, or a number that is not in its fact, is refused and stops the assembly.

In [ ]:
# ── Cell 1: settings ─────────────────────────────────────────────────────────
VIDEO = ""  # @param {type:"string"}
# Empty = the newest video with a voice and no finished render. Else the folder name: "01_uber".

MODE = "extract"  # @param ["plan", "extract", "full"]
# plan: check and time everything, no render · extract: the scenes below only · full: the whole video

SCENES = "1-10"  # @param {type:"string"}
# With MODE = "extract": first-last scene numbers.

ROOT = "/content/drive/MyDrive/WhoseDime"  # @param {type:"string"}
# The WhoseDime folder in your Drive. If it is "shared with me", add a shortcut to it in "My Drive".

In [ ]:
# ── Cell 2: code and Drive ──────────────────────────────────────────────────
import os, subprocess, sys
if not globals().get("REPO_READY"):  # opened directly, not through the launcher: take main (D5)
    subprocess.run(["rm", "-rf", "/content/Channel"], check=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                       "https://github.com/Johnfrancis6/Channel.git", "/content/Channel"]).returncode:
        raise SystemExit("❌ Could not fetch the code (branch main).")
sys.path.insert(0, "/content/Channel/whosedime/notebooks")
sys.path.insert(0, "/content/Channel/whosedime/whosedime-video/scripts")
for name in ("assemble", "charts", "checker", "voice_whosedime", "launcher"):
    sys.modules.pop(name, None)
import launcher
import assemble
COMMIT = globals().get("COMMIT") or launcher.commit()
print(f"✓ Code: main @ {COMMIT}")
for tool in ("ffmpeg", "ffprobe"):
    if subprocess.run([tool, "-version"], capture_output=True).returncode:
        raise SystemExit(f"❌ {tool} is missing")

from pathlib import Path
ROOT = Path(ROOT)
if not ROOT.is_dir() and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
if not ROOT.is_dir():
    raise SystemExit(f"❌ No WhoseDime folder at {ROOT}. Fix ROOT in cell 1.")

In [ ]:
# ── Cell 3: the video folder ────────────────────────────────────────────────
try:
    FOLDER = assemble.find_video(ROOT, VIDEO)
    ONLY = assemble.parse_range(SCENES) if MODE == "extract" else None
except (assemble.AssemblyError, ValueError) as e:
    raise SystemExit(f"❌ {e}")
if MODE not in ("plan", "extract", "full"):
    raise SystemExit(f"❌ MODE '{MODE}' is not plan, extract or full")
print(f"🎬 Video: {FOLDER.name}  ·  mode {MODE}" + (f" (scenes {ONLY[0]}-{ONLY[1]})" if ONLY else ""))

In [ ]:
# ── Cell 4: plan, then render on the Colab disk, then copy to render/ ───────
WORK = "/content/render" if os.path.isdir("/content") else None  # Drive is slow to write a video frame by frame
try:
    RESULT = assemble.run(FOLDER, ONLY, plan_only=(MODE == "plan"), work_dir=WORK)
except assemble.AssemblyError as e:
    raise SystemExit(f"❌ {e}")
if MODE == "plan":
    print("\n✅ Plan done: render/chapters.txt and render/plan.json written. Next: MODE = 'extract'.")
else:
    print(f"\n{'✅' if RESULT['ready'] else '❌'} {RESULT['output']} in Videos/{FOLDER.name}/render/ — "
          f"ask Claude to read {RESULT['report'].replace('report_', 'verdict_')}.")

In [ ]:
# ── Cell 5: the preview frames ──────────────────────────────────────────────
try:
    from IPython.display import Image as Show, display
except ImportError:  # outside Colab and Jupyter: the frames are in render/ all the same
    display = None
if MODE != "plan" and display:
    for frame in sorted((FOLDER / "render" / RESULT["preview"]).glob("*.jpg")):
        print(frame.name)
        display(Show(str(frame), width=480))